# MNIST · Red Densa + Optuna + MLflow + Regularización

**Actividad**

1. Diseñar una red **densa secuencial (no convolucional)** para clasificar dígitos MNIST en Keras.
   - (a) Usar **Optuna** para buscar una buena arquitectura (nº de capas, neuronas, funciones de activación, optimizador). **Sin regularización** en esta etapa. Registrar en **MLflow**.
   - (b) Con la **mejor red**, entrenar con los mismos datos usando regularizaciones: **L1, L2, L1-L2, Dropout, Dropout+L1-L2**. Comentar cada caso y responder si la regularización ayudó a mejorar la eficiencia antes del sobreajuste. Registrar en **MLflow**.

> Los resultados numéricos se generan al ejecutar el notebook.

## 0. Instalar dependencias

In [ ]:
!pip -q install optuna mlflow dagshub

## 1. Importaciones y configuración

In [ ]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

import optuna
import mlflow

from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Optuna:", optuna.__version__)
print("MLflow:", mlflow.__version__)
print("Dispositivos:", tf.config.list_physical_devices())

## 2. Cargar y preparar MNIST

Normalizamos a [0, 1] y separamos un 10% de entrenamiento como validación (reproducible).

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

# Split reproducible: 90% train / 10% validation
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(x_train))
n_val = int(0.10 * len(x_train))
val_idx, train_idx = idx[:n_val], idx[n_val:]

x_val, y_val = x_train[val_idx], y_train[val_idx]
x_tr,  y_tr  = x_train[train_idx], y_train[train_idx]

print("Train:", x_tr.shape)
print("Validation:", x_val.shape)
print("Test:", x_test.shape)

## 3. Visualizar ejemplos

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.ravel(), x_tr[:10], y_tr[:10]):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Clase: {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Constructor de la red densa

Red **densa secuencial**: cada imagen 28×28 se aplana (`Flatten`) a 784 entradas.
La misma función sirve para el inciso (a) —sin regularización— y para el (b) —con L1/L2/L1-L2 y/o Dropout—.

In [ ]:
def build_dense_model(n_layers, units, activations, optimizer_name,
                      learning_rate, regularization=None, dropout_rate=0.0):
    """Construye una red densa configurable.

    regularization: None | "l1" | "l2" | "l1_l2"
    dropout_rate:   0.0 desactiva Dropout
    """
    model = keras.Sequential(name="MNIST_Dense")
    model.add(layers.Input(shape=(28, 28)))
    model.add(layers.Flatten())

    for i in range(n_layers):
        reg = None
        if regularization == "l1":
            reg = regularizers.l1(1e-4)
        elif regularization == "l2":
            reg = regularizers.l2(1e-4)
        elif regularization == "l1_l2":
            reg = regularizers.l1_l2(l1=1e-4, l2=1e-4)

        model.add(layers.Dense(units[i], activation=activations[i],
                               kernel_regularizer=reg))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(10, activation="softmax"))

    opt = {
        "adam":    keras.optimizers.Adam(learning_rate=learning_rate),
        "rmsprop": keras.optimizers.RMSprop(learning_rate=learning_rate),
        "sgd":     keras.optimizers.SGD(learning_rate=learning_rate),
    }[optimizer_name]

    model.compile(optimizer=opt,
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

## 5. Configurar MLflow

Se registra en **MLflow**. Para obtener el **enlace del servidor** que pide el reporte se usa **DagsHub** (servidor MLflow remoto conectado a tu repo de GitHub).

Pasos:
1. Crea cuenta y repo en https://dagshub.com (conecta tu repo de GitHub).
2. DagsHub → *Settings → Tokens* → copia tu token.
3. Pega el token abajo (o expórtalo como variable de entorno `DAGSHUB_TOKEN`).

Si no defines el token, se usa MLflow local (`./mlruns`) para poder ejecutar sin conexión.

In [ ]:
DAGSHUB_USER  = "BrendaMonesA"
DAGSHUB_REPO  = "Redes_Neuronales_Artificiales"
DAGSHUB_TOKEN = os.environ.get("DAGSHUB_TOKEN", "")   # <-- pega aqui tu token de DagsHub

if DAGSHUB_TOKEN:
    os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
    os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
    mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow")
    print("MLflow -> DagsHub (remoto).")
    print("ENLACE DEL SERVIDOR (para el reporte):")
    print(f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}/experiments")
else:
    mlflow.set_tracking_uri("file:./mlruns")
    print("MLflow -> local ./mlruns  (define DAGSHUB_TOKEN para el enlace publico)")

mlflow.set_experiment("MNIST_Dense_Optuna")
print("Tracking URI:", mlflow.get_tracking_uri())

## 6. Función objetivo de Optuna — inciso (a)

Se explora **sin regularización**:
- nº de capas: 1–4
- neuronas por capa: 32, 64, 128, 256, 512
- activación: relu, tanh, elu
- optimizador: adam, rmsprop, sgd
- learning rate: 1e-4 a 1e-2 (log)
- batch size: 32, 64, 128

Cada trial se registra como un run en MLflow.

In [ ]:
N_TRIALS = 30
EPOCHS_OPTUNA = 10

def objective(trial):
    n_layers = trial.suggest_int("n_layers", 1, 4)
    units = [trial.suggest_categorical(f"units_{i}", [32, 64, 128, 256, 512])
             for i in range(n_layers)]
    activations = [trial.suggest_categorical(f"activation_{i}", ["relu", "tanh", "elu"])
                   for i in range(n_layers)]
    optimizer_name = trial.suggest_categorical("optimizer", ["adam", "rmsprop", "sgd"])
    learning_rate  = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    batch_size     = trial.suggest_categorical("batch_size", [32, 64, 128])

    model = build_dense_model(n_layers, units, activations,
                              optimizer_name, learning_rate)   # sin regularizacion

    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}"):
        mlflow.log_params(trial.params)
        history = model.fit(x_tr, y_tr, validation_data=(x_val, y_val),
                            epochs=EPOCHS_OPTUNA, batch_size=batch_size, verbose=0)
        best_val_acc = float(max(history.history["val_accuracy"]))
        mlflow.log_metric("best_val_accuracy", best_val_acc)
        mlflow.log_metric("final_val_loss", float(history.history["val_loss"][-1]))

    keras.backend.clear_session()
    return best_val_acc

## 7. Ejecutar la búsqueda con Optuna

In [ ]:
study = optuna.create_study(direction="maximize", study_name="MNIST_Dense_Search")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Mejor validation accuracy:", study.best_value)
print("\nMejores hiperparametros:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

## 8. Tabla y gráficas de Optuna

In [ ]:
trials_df = study.trials_dataframe(attrs=("number", "value", "params", "state"))
display(trials_df.sort_values("value", ascending=False).head(10))

try:
    optuna.visualization.matplotlib.plot_optimization_history(study); plt.show()
    optuna.visualization.matplotlib.plot_param_importances(study);    plt.show()
except Exception as e:
    print("No se pudieron mostrar las graficas de Optuna:", e)

## 9. Reconstruir la mejor arquitectura

In [ ]:
bp = study.best_params
best_n_layers   = bp["n_layers"]
best_units      = [bp[f"units_{i}"] for i in range(best_n_layers)]
best_activations= [bp[f"activation_{i}"] for i in range(best_n_layers)]
best_optimizer  = bp["optimizer"]
best_lr         = bp["learning_rate"]
best_batch_size = bp["batch_size"]

print("Mejor arquitectura encontrada")
print("  Capas:", best_n_layers)
print("  Neuronas:", best_units)
print("  Activaciones:", best_activations)
print("  Optimizador:", best_optimizer)
print("  Learning rate:", best_lr)
print("  Batch size:", best_batch_size)
print("  Val accuracy:", study.best_value)

## 10. Entrenar los modelos finales — inciso (b)

Se reutiliza la mejor arquitectura y se entrena con los mismos datos, variando la regularización.

In [ ]:
EPOCHS_FINAL = 20

def train_model(regularization=None, dropout_rate=0.0, name="model"):
    keras.backend.clear_session()
    model = build_dense_model(best_n_layers, best_units, best_activations,
                              best_optimizer, best_lr,
                              regularization=regularization, dropout_rate=dropout_rate)

    with mlflow.start_run(run_name=name):
        mlflow.log_param("model_type", name)
        mlflow.log_param("regularization", str(regularization))
        mlflow.log_param("dropout_rate", dropout_rate)

        history = model.fit(x_tr, y_tr, validation_data=(x_val, y_val),
                            epochs=EPOCHS_FINAL, batch_size=best_batch_size, verbose=1)
        test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

        mlflow.log_metric("test_loss", float(test_loss))
        mlflow.log_metric("test_accuracy", float(test_acc))

    return model, history, test_loss, test_acc

## 11. Entrenar los seis casos (Base + 5 regularizaciones)

In [ ]:
configs = {
    "Base":            (None,    0.0),
    "L1":              ("l1",    0.0),
    "L2":              ("l2",    0.0),
    "L1-L2":           ("l1_l2", 0.0),
    "Dropout":         (None,    0.30),
    "Dropout + L1-L2": ("l1_l2", 0.30),
}

results, models = {}, {}
for name, (reg, dr) in configs.items():
    print(f"\n===== {name} =====")
    model, history, test_loss, test_acc = train_model(regularization=reg, dropout_rate=dr, name=name)
    models[name] = model
    results[name] = {"history": history, "test_loss": test_loss, "test_accuracy": test_acc}

print("\nTodos los modelos fueron entrenados.")

## 12. Tabla comparativa

In [ ]:
rows = []
for name, r in results.items():
    h = r["history"].history
    train_acc = max(h["accuracy"])
    val_acc   = max(h["val_accuracy"])
    rows.append({
        "Modelo": name,
        "Mejor Train Acc": train_acc,
        "Mejor Val Acc": val_acc,
        "Test Acc": r["test_accuracy"],
        "Test Loss": r["test_loss"],
        "Min Val Loss": min(h["val_loss"]),
        "Brecha Train-Val": train_acc - val_acc,
    })

results_df = pd.DataFrame(rows)
display(results_df.sort_values("Test Acc", ascending=False).style.format({
    "Mejor Train Acc": "{:.4f}", "Mejor Val Acc": "{:.4f}", "Test Acc": "{:.4f}",
    "Test Loss": "{:.4f}", "Min Val Loss": "{:.4f}", "Brecha Train-Val": "{:.4f}"}))

## 13. Curvas de Validation Accuracy

In [ ]:
plt.figure(figsize=(12, 7))
for name, r in results.items():
    plt.plot(r["history"].history["val_accuracy"], label=name)
plt.xlabel("Epoca"); plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy por modelo"); plt.legend(); plt.grid(True, alpha=0.3); plt.show()

## 14. Curvas de Validation Loss

In [ ]:
plt.figure(figsize=(12, 7))
for name, r in results.items():
    plt.plot(r["history"].history["val_loss"], label=name)
plt.xlabel("Epoca"); plt.ylabel("Validation Loss")
plt.title("Validation Loss por modelo"); plt.legend(); plt.grid(True, alpha=0.3); plt.show()

## 15. Modelo base: Train vs Validation

In [ ]:
h = results["Base"]["history"].history

plt.figure(figsize=(10, 6))
plt.plot(h["accuracy"], label="Train Acc"); plt.plot(h["val_accuracy"], label="Val Acc")
plt.xlabel("Epoca"); plt.ylabel("Accuracy"); plt.title("Base: Train vs Val Accuracy")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

plt.figure(figsize=(10, 6))
plt.plot(h["loss"], label="Train Loss"); plt.plot(h["val_loss"], label="Val Loss")
plt.xlabel("Epoca"); plt.ylabel("Loss"); plt.title("Base: Train vs Val Loss")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

## 16. Comparación de Test Accuracy

In [ ]:
plot_df = results_df.sort_values("Test Acc")
plt.figure(figsize=(10, 6))
plt.barh(plot_df["Modelo"], plot_df["Test Acc"])
plt.xlabel("Test Accuracy"); plt.title("Test Accuracy por modelo")
plt.xlim(max(0, plot_df["Test Acc"].min() - 0.02), min(1, plot_df["Test Acc"].max() + 0.01))
plt.grid(axis="x", alpha=0.3); plt.show()

## 17. Análisis de sobreajuste

La **brecha Train − Val** es un indicador descriptivo del sobreajuste: cuanto mayor, más memoriza el modelo el entrenamiento sin generalizar. Debe leerse junto con las curvas de loss y el desempeño en test.

In [ ]:
display(results_df[["Modelo", "Mejor Train Acc", "Mejor Val Acc", "Test Acc", "Brecha Train-Val"]]
        .sort_values("Brecha Train-Val").style.format({
            "Mejor Train Acc": "{:.4f}", "Mejor Val Acc": "{:.4f}",
            "Test Acc": "{:.4f}", "Brecha Train-Val": "{:.4f}"}))

## 18. Exportar resultados y modelos

In [ ]:
results_df.to_csv("resultados_regularizacion.csv", index=False)
study.trials_dataframe().to_csv("optuna_trials.csv", index=False)

os.makedirs("modelos_mnist", exist_ok=True)
for name, model in models.items():
    safe = name.lower().replace(" ", "_").replace("+", "plus")
    model.save(f"modelos_mnist/{safe}.keras")

print("Generados: resultados_regularizacion.csv, optuna_trials.csv, modelos_mnist/")

# 19. Texto guía para el reporte

Completa con tus resultados reales tras ejecutar el notebook.

### (a) Optuna
- Nº de trials, hiperparámetros explorados, mejor arquitectura y su validation accuracy.
- Enlace del servidor MLflow (DagsHub) con las gráficas.

### (b) Regularización
Para cada caso (L1, L2, L1-L2, Dropout, Dropout+L1-L2) compara vs. Base:
train acc, val acc, test acc, val loss, brecha train-val y forma de las curvas.

### Pregunta principal
**¿La regularización ayudó a mejorar la eficiencia antes del sobreajuste?**
Argumenta con tus datos: si una técnica reduce la brecha train-val manteniendo o mejorando
val/test, ayudó a generalizar; si empeora el desempeño, indícalo. No asumas resultados antes de observarlos.

## 20. Visualizar los experimentos en MLflow

- **DagsHub:** abre `https://dagshub.com/<TU_USUARIO>/Redes_Neuronales_Artificiales/experiments`.
- **Local:** ejecuta la celda siguiente y abre `http://localhost:5000`.

In [ ]:
# Solo si usaste MLflow local:
!nohup mlflow ui --backend-store-uri ./mlruns --host 0.0.0.0 --port 5000 > mlflow.log 2>&1 &
print("MLflow UI local en http://localhost:5000")